In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import fiona
import pyproj
from osgeo import gdal
from area import area

In [ ]:
def shp_to_gdf(shapefile_path, driver):
    shapefile_features =[]
    with fiona.collection(shapefile_path, driver=driver) as shp_in:
        shp_crs = pyproj.crs.CRS(shp_in.crs)
        shpschema = shp_in.schema.copy()
        for obj in shp_in:
            shapefile_features.append(obj)
    shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)
    shp_gdf['bounds'] = shp_gdf.geometry.apply(lambda x: x.bounds)
    shp_gdf['bounds'] = [list(i) for i in shp_gdf['bounds']]
    return shp_gdf

def get_intersection_area(poly_1, poly_2):
    if not poly_1.is_valid:
        poly_1 = poly_1.buffer(0)
    if not poly_2.is_valid:
        poly_2 = poly_2.buffer(0)
    return (poly_1.intersection(poly_2).area)/10**4

In [ ]:
data_path = "../data/all_projects_TM_20240614/all_projects_TM.shp"

In [ ]:
shapefile_features =[]
with fiona.collection(data_path, 'r') as shp_in:
    shp_crs = pyproj.crs.CRS(shp_in.crs)
    shpschema = shp_in.schema.copy()
    for obj in shp_in:
        shapefile_features.append(obj)
shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)
#shp_gdf['bounds'] = shp_gdf.geometry.apply(lambda x: x.bounds)
#shp_gdf['bounds'] = [list(i) for i in shp_gdf['bounds']]
shp_gdf['plantstart'] = pd.to_datetime(shp_gdf['plantstart'], format='mixed').dt.date

In [ ]:
project_list = shp_gdf['Project'].unique()

In [ ]:
df_short_list = []
for project in project_list:
    project_gdf = shp_gdf[shp_gdf['Project'] == project]
    project_gdf['Project'] = project
    project_gdf_copy = project_gdf.copy()
    project_gdf_copy['geometry'] = project_gdf_copy['geometry'].to_crs({'init': 'epsg:3857'})
    project_gdf['Area_ha'] = project_gdf_copy['geometry'].apply(lambda x: x.area/10**4)
    df_short_list.append(project_gdf)

all_proj_df = gpd.GeoDataFrame(pd.concat(df_short_list, ignore_index=True))
keep_columns = ['geometry', 'Project', 'SiteName', 'poly_name', 'plantstart',
        'target_sys', 'practice', 'Area_ha']
#long_colnames_keep = [item for item in all_proj_df.columns if item in keep_columns]
all_proj_df= all_proj_df[keep_columns]
all_proj_df = all_proj_df[~pd.isna(all_proj_df['poly_name'])]

In [ ]:
all_proj_df

In [ ]:
colnames = all_proj_df.columns
remove = {'target_sys', 'Area_ha'}
new_colnames = [item for item in colnames if item not in remove ]
project_gdf_long = pd.pivot(all_proj_df, index = new_colnames, columns='target_sys', values='Area_ha').reset_index()

# Imagery availablity 

In [ ]:
imagery_data_path = "../data/afr100_cohort1_imagery_availability/"
imagery_files = os.listdir(imagery_data_path)
all_project_imagery = []
for project in imagery_files:
    project_df = pd.read_csv(f"{imagery_data_path}/{project}")
    project_sub_df = project_df[['Name', 'properties.datetime','collection', 'properties.eo:cloud_cover', 'properties.off_nadir_avg']]
    project_sub_df['Project'] = project.replace('afr100_', '').replace('_imagery_availability.csv', '')
    all_project_imagery.append(project_sub_df)
all_projects_imagery_df = pd.concat(all_project_imagery).reset_index()
all_projects_imagery_df = all_projects_imagery_df[['Project', 'Name', 'properties.datetime','collection', 'properties.eo:cloud_cover', 'properties.off_nadir_avg']]
all_projects_imagery_df = all_projects_imagery_df[~pd.isna(all_projects_imagery_df['Name'])]

In [ ]:
all_projects_imagery_df.rename(columns={'Name':'poly_name'}, inplace=True)
all_projects_imagery_df['properties.datetime'] =pd.to_datetime(all_projects_imagery_df['properties.datetime'], format='mixed').dt.normalize()
all_projects_imagery_df['properties.datetime'] =all_projects_imagery_df['properties.datetime'].apply(lambda x: x.replace(tzinfo=None))

In [ ]:
all_proj_df_imagery = all_proj_df.merge(all_projects_imagery_df, on=['Project', 'poly_name'], how='left')

In [ ]:
all_proj_df_imagery = all_proj_df_imagery[~pd.isna(all_proj_df_imagery['plantstart'])]
all_proj_df_imagery = all_proj_df_imagery[all_proj_df_imagery['plantstart'] != '15-04-2024 - 15-05-2024']
all_proj_df_imagery['plantstart'] =pd.to_datetime(all_proj_df_imagery['plantstart'], format='mixed')
all_proj_df_imagery['properties.datetime'] =pd.to_datetime(all_proj_df_imagery['properties.datetime'], format='mixed').dt.normalize()
all_proj_df_imagery['properties.datetime'] =all_proj_df_imagery['properties.datetime'].apply(lambda x: x.replace(tzinfo=None))
all_proj_df_imagery['dateDiff']=(all_proj_df_imagery['properties.datetime'] - all_proj_df_imagery['plantstart']).dt.days

# Slope/Aspect

In [ ]:
slope_df = pd.read_csv("../data/bucketing_output/polygon_slope_aspect.csv")
slope_df = slope_df[['Project', 'poly_name', 'slope_stats', 'aspect_stats']]
slope_df.shape

In [ ]:
all_proj_df_imagery_slope = all_proj_df_imagery.merge(slope_df, on=['Project', 'poly_name'], how='left')

In [ ]:
all_proj_df_imagery_slope

# Landscape boundaries

In [ ]:
landscape_data_path = "../data/landscape_boundaries/"
landscape_files = os.listdir(landscape_data_path)

In [ ]:
landscape_list = ['KRB.shp','GRV.shp', 'GhanaCocoaBelt.shp']
landscape_poly_list = []
for landscape in landscape_list:
    shapefile_features =[]
    with fiona.collection(f"{landscape_data_path}/{landscape}", driver='ESRI Shapefile') as shp_in:
        shp_crs = pyproj.crs.CRS(shp_in.crs)
        shpschema = shp_in.schema.copy()
        for obj in shp_in:
            shapefile_features.append(obj)
    shp_gdf = gpd.GeoDataFrame.from_features([feature for feature in shapefile_features], crs=shp_crs)
    shp_gdf['Landscape'] = landscape.split('.')[0]
    landscape_poly_list.append(shp_gdf)
all_landscape_df = gpd.GeoDataFrame(pd.concat(landscape_poly_list, ignore_index=True))

In [ ]:
all_landscape_df_sub = all_landscape_df[all_landscape_df.index.isin([0,1,5])].reset_index()
all_landscape_df_sub['geometry'] =all_landscape_df_sub['geometry'].to_crs({'init': 'epsg:3857'})